# Thu thập giá OHLCV (`ingest_price_data`) — Luồng 1

**Đồ án:** Hệ thống Data Pipeline và tra cứu phân tích thị trường chứng khoán Việt Nam đa nguồn.

**Mục đích:** Tải lịch sử giá OHLCV có cấu trúc qua thư viện **vnstock** (free tier), chuẩn hóa/làm sạch, ghi vào **Data Lake** dạng Parquet theo partition Hive-style `date=YYYY-MM-DD` (ngày chạy pipeline).

**Nguồn `Quote`:** ưu tiên **KBS** (`PRIMARY_QUOTE_SOURCE`), fallback **VCI** (`FALLBACK_QUOTE_SOURCE`). Nguồn **TCBS** đã **deprecated** trong hệ vnstock; không dùng trong code.

**Đầu ra:** `../data-lake/raw/price/date=<YYYY-MM-DD>/<Mã>.parquet` (một file mỗi mã).

**Cách chạy:** Chọn kernel Python của project (venv), chạy lần lượt các ô: import → helper → định nghĩa hàm → `main()` → ô kiểm tra dữ liệu cuối cùng.


In [ ]:
# vnstock Quote: PRIMARY = KBS, FALLBACK = VCI (nguồn ổn định).
# TCBS đã deprecated theo tài liệu vnstock (CLAUDE.md) — không dùng làm source.
import logging
import os
import time
from datetime import date
from pathlib import Path

import pandas as pd
from vnstock import Quote

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

TICKERS: list[str] = [
    "ACB",
    "BCM",
    "BID",
    "BVH",
    "CTG",
    "FPT",
    "GAS",
    "GVR",
    "HDB",
    "HPG",
    "MBB",
    "MSN",
    "MWG",
    "PLX",
    "POW",
    "SAB",
    "SHB",
    "SSI",
    "STB",
    "TCB",
    "TPB",
    "VCB",
    "VHM",
    "VIB",
    "VIC",
    "VJC",
    "VNM",
    "VPB",
    "VRE",
    "VGC",
    "DGC",
    "DPM",
    "DCM",
    "HSG",
    "NKG",
    "KDC",
    "QNS",
    "SCS",
    "ACV",
    "VTP",
    "KBC",
    "NVL",
    "PDR",
    "DXG",
    "KDH",
    "AGG",
    "HDG",
    "REE",
    "PNJ",
    "GMD",
]
PRIMARY_QUOTE_SOURCE = "kbs"
FALLBACK_QUOTE_SOURCE = "vci"

assert len(TICKERS) == 50
assert len(set(TICKERS)) == len(TICKERS)

In [6]:
def _ensure_ingestion_workdir() -> None:
    """Đặt cwd là thư mục chứa notebook để ../data-lake đúng partition."""
    cwd = Path.cwd().resolve()
    if (cwd / "ingest_price_data.ipynb").is_file():
        target = cwd
    elif (cwd / "ingestion" / "ingest_price_data.ipynb").is_file():
        target = cwd / "ingestion"
    else:
        target = cwd
    os.chdir(str(target))


def _date_years_ago(d: date, years: int) -> date:
    try:
        return d.replace(year=d.year - years)
    except ValueError:
        return d.replace(year=d.year - years, day=28)

In [7]:
def fetch_ticker_data(ticker: str, start: str, end: str) -> pd.DataFrame:
    """
    Tải OHLCV qua vnstock Quote: thử PRIMARY_QUOTE_SOURCE (KBS), rồi FALLBACK_QUOTE_SOURCE (VCI).
    """
    sym = ticker.upper().strip()
    sources: tuple[str, ...] = (PRIMARY_QUOTE_SOURCE, FALLBACK_QUOTE_SOURCE)
    last_err: Exception | None = None

    for src in sources:
        try:
            quote = Quote(source=src, symbol=sym)
            df = quote.history(start=start, end=end, interval="1D")
            if df is not None and not df.empty:
                logger.info("Đã lấy dữ liệu %s từ nguồn %s", sym, src.upper())
                return df.copy()
        except Exception as e:
            last_err = e
            logger.debug("Quote source=%s symbol=%s thất bại: %s", src, sym, e)
            continue

    if last_err is not None:
        logger.warning(
            "Không lấy được dữ liệu %s sau khi thử các nguồn: %s",
            sym,
            last_err,
        )
    return pd.DataFrame()


def transform_data(df: pd.DataFrame) -> pd.DataFrame:
    """Chuẩn hóa cột, time, lọc volume, quy đổi giá nghìn đồng nếu cần, cờ bất thường."""
    if df.empty:
        return df.copy()

    out = df.copy()
    rename_map: dict[str, str] = {}
    for col in out.columns:
        key = str(col).strip().lower().replace(" ", "")
        if key in ("tradingdate", "trading_date", "date", "ngay"):
            rename_map[col] = "time"
        else:
            rename_map[col] = key
    out = out.rename(columns=rename_map)

    required = {"time", "open", "high", "low", "close", "volume"}
    missing = required - set(out.columns)
    if missing:
        raise KeyError(f"Thiếu cột sau khi chuẩn hóa: {missing}")

    out = out.loc[:, list(required)]
    out["time"] = pd.to_datetime(out["time"], errors="coerce")
    out = out.dropna(subset=["time"])

    out["_vol_num"] = pd.to_numeric(out["volume"], errors="coerce")
    out = out.loc[(out["_vol_num"].notna()) & (out["_vol_num"] > 0)].copy()
    out["volume"] = out["_vol_num"].astype("int64")
    out = out.drop(columns=["_vol_num"])

    for col in ("open", "high", "low", "close"):
        out[col] = pd.to_numeric(out[col], errors="coerce")

    ohlc_max = out[["open", "high", "low", "close"]].max(axis=1, numeric_only=True).max()
    if pd.notna(ohlc_max) and float(ohlc_max) > 10_000:
        for col in ("open", "high", "low", "close"):
            out[col] = out[col] / 1000.0

    out["is_suspicious"] = (out["close"] < 0) | (out["high"] < out["low"])
    out = out.sort_values("time", ascending=True).reset_index(drop=True)
    return out


def save_to_datalake(df: pd.DataFrame, ticker: str, run_date: str) -> str:
    """Ghi Parquet vào ../data-lake/raw/price/date=RUN_DATE/TICKER.parquet."""
    base_dir = os.path.abspath(
        os.path.join(os.getcwd(), "..", "data-lake", "raw", "price")
    )
    partition = os.path.join(base_dir, f"date={run_date}")
    os.makedirs(partition, exist_ok=True)
    safe_ticker = ticker.upper().strip()
    path = os.path.join(partition, f"{safe_ticker}.parquet")
    try:
        df.to_parquet(path, engine="pyarrow", index=False)
    except ImportError:
        df.to_parquet(path, engine="fastparquet", index=False)
    logger.info("Đã ghi %s dòng → %s", len(df), path)
    return path

In [ ]:
def main() -> None:
    _ensure_ingestion_workdir()
    run_dt = date.today()
    run_date = run_dt.isoformat()
    start_dt = _date_years_ago(run_dt, 3)
    start_s = start_dt.isoformat()
    end_s = run_dt.isoformat()

    logger.info(
        "Partition date=%s | cửa sổ vnstock history %s → %s",
        run_date,
        start_s,
        end_s,
    )

    total = len(TICKERS)
    for idx, ticker in enumerate(TICKERS):
        logger.info("[%s/%s] Đang tải %s...", idx + 1, total, ticker)
        try:
            raw = fetch_ticker_data(ticker, start_s, end_s)
            if raw.empty:
                logger.warning("Bỏ qua %s — không có dữ liệu thô", ticker)
            else:
                cleaned = transform_data(raw)
                if cleaned.empty:
                    logger.warning("Bỏ qua %s — sau làm sạch không còn dòng", ticker)
                else:
                    save_to_datalake(cleaned, ticker, run_date)
                    logger.info(
                        "Thành công %s (%s dòng)",
                        ticker,
                        len(cleaned),
                    )
        except Exception:
            logger.exception("Lỗi khi xử lý mã %s — tiếp tục mã kế tiếp", ticker)

        if idx < total - 1:
            time.sleep(4)

    logger.info("Hoàn tất pipeline cho %s mã.", total)


main()

2026-04-03 02:01:50 [INFO] Partition date=2026-04-03 | cửa sổ vnstock history 2023-04-03 → 2026-04-03
2026-04-03 02:01:50 [INFO] [1/50] Đang tải ACB...


2026-04-03 02:01:51 [INFO] Đã lấy dữ liệu ACB từ nguồn KBS
2026-04-03 02:01:51 [INFO] Đã ghi 749 dòng → C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\price\date=2026-04-03\ACB.parquet
2026-04-03 02:01:51 [INFO] Thành công ACB (749 dòng)
2026-04-03 02:01:52 [INFO] [2/50] Đang tải BCM...
2026-04-03 02:01:53 [INFO] Đã lấy dữ liệu BCM từ nguồn KBS
2026-04-03 02:01:53 [INFO] Đã ghi 749 dòng → C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\price\date=2026-04-03\BCM.parquet
2026-04-03 02:01:53 [INFO] Thành công BCM (749 dòng)
2026-04-03 02:01:54 [INFO] [3/50] Đang tải BID...
2026-04-03 02:01:54 [INFO] Đã lấy dữ liệu BID từ nguồn KBS
2026-04-03 02:01:54 [INFO] Đã ghi 749 dòng → C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\data-lake\raw\price\date=2026-04-03\BID.parquet
2026-04-03 02:01:54 [INFO] Thành công BID (749 dòng)
2026-04-03 02:01:55 [INFO] [4/50] Đang tải BVH...
2026-04-03 02:01:55 [INFO] Đã lấy dữ liệu BVH từ nguồn KBS
2026-04-03 02:01:55 [INFO] Đã ghi 7


⚠️ 
⚠️  GIỚI HẠN API ĐÃ ĐẠT TỐI ĐA (Rate Limit Exceeded)

📌 Bạn đã đạt giới hạn tối đa số lượt yêu cầu API trong 1 phút (minute).
   (You have reached the maximum API request limit for this period)

📊 Chi tiết (Details):
   • Gói hiện tại: Khách (Guest)
   • Giới hạn: 20 requests/phút
   • Đã sử dụng: 20/20
   • Chờ 40 giây để tiếp tục (Wait to retry)

💡 Giải pháp (Solutions):
   1️⃣ Chờ 40 giây rồi thử lại
      (Wait and retry)
   2️⃣ Tham gia gói thành viên tài trợ để sử dụng không bị gián đoạn
      (Join sponsor membership for uninterrupted access)

🚀 Nâng cấp (Upgrade):
   • Phiên bản cộng đồng (60 request/phút - Community):
     Đăng ký API key miễn phí: https://vnstocks.com/login
   • Gói thành viên tài trợ (180-600 request/phút - Sponsor):
     Tham gia: https://vnstocks.com/insiders-program
     Sau khi tham gia tài trợ, cài bộ thư viện riêng vnstock_data theo hướng dẫn https://vnstocks.com/onboard-member





SystemExit: Rate limit exceeded. 
============================================================
⚠️  GIỚI HẠN API ĐÃ ĐẠT TỐI ĐA (Rate Limit Exceeded)
============================================================

📌 Bạn đã đạt giới hạn tối đa số lượt yêu cầu API trong 1 phút (minute).
   (You have reached the maximum API request limit for this period)

📊 Chi tiết (Details):
   • Gói hiện tại: Khách (Guest)
   • Giới hạn: 20 requests/phút
   • Đã sử dụng: 20/20
   • Chờ 40 giây để tiếp tục (Wait to retry)

💡 Giải pháp (Solutions):
   1️⃣ Chờ 40 giây rồi thử lại
      (Wait and retry)
   2️⃣ Tham gia gói thành viên tài trợ để sử dụng không bị gián đoạn
      (Join sponsor membership for uninterrupted access)

🚀 Nâng cấp (Upgrade):
   • Phiên bản cộng đồng (60 request/phút - Community):
     Đăng ký API key miễn phí: https://vnstocks.com/login
   • Gói thành viên tài trợ (180-600 request/phút - Sponsor):
     Tham gia: https://vnstocks.com/insiders-program
     Sau khi tham gia tài trợ, cài b

To exit: use 'exit', 'quit', or Ctrl-D.


In [ ]:
# Kiểm tra dữ liệu sau khi chạy `main()` — có thể chạy riêng ô này sau khi đã import `pandas` và chạy ô `_ensure_ingestion_workdir` (hoặc dùng logic path tương đương bên dưới).

import pandas as pd
from pathlib import Path


def _datalake_price_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "ingest_price_data.ipynb").is_file():
        base = cwd
    elif (cwd / "ingestion" / "ingest_price_data.ipynb").is_file():
        base = cwd / "ingestion"
    else:
        base = cwd
    return (base / ".." / "data-lake" / "raw" / "price").resolve()


price_root = _datalake_price_root()
partitions = sorted(
    p for p in price_root.glob("date=*") if p.is_dir()
)
if not partitions:
    raise FileNotFoundError(f"Không thấy partition date=* dưới {price_root}")

latest = partitions[-1]
print(f"[QC] Partition mới nhất: {latest.name}")

parquet_files = sorted(latest.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"Không có file .parquet trong {latest}")

frames: list[pd.DataFrame] = []
for fpath in parquet_files:
    ticker = fpath.stem.upper()
    sub = pd.read_parquet(fpath)
    sub = sub.copy()
    sub["ticker"] = ticker
    frames.append(sub)

df_qc = pd.concat(frames, ignore_index=True)

n_ma = df_qc["ticker"].nunique()
n_rows = len(df_qc)
n_susp = int(df_qc["is_suspicious"].sum()) if "is_suspicious" in df_qc.columns else 0

print(f"[QC] Số mã (ticker) duy nhất: {n_ma}")
print(f"[QC] Tổng số dòng: {n_rows}")
print(f"[QC] Số dòng is_suspicious=True: {n_susp}")
print("[QC] dtypes:")
print(df_qc.dtypes)
print("[QC] df.head(3) (tổng hợp nhiều mã):")
print(df_qc.head(3))
sample_ticker = sorted(df_qc["ticker"].unique())[0]
print(f"[QC] head(3) mã mẫu {sample_ticker}:")
print(df_qc.loc[df_qc["ticker"] == sample_ticker].head(3))
